In [15]:
# Инициализация и логирование
import os
import umap
import torch
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import torch.nn as nn
import torch.optim as optim
from datetime import datetime
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from IPython.display import display
from sklearn.svm import OneClassSVM
from sklearn.neighbors import KernelDensity
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

debug = 1  # 1 — выводить отладочные логи, 0 — скрывать
warnings.filterwarnings("ignore", message=".*n_jobs value 1 overridden.*")

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("hardtype/rbc-economy-news")

print("Path to dataset files:", path)

100%|██████████| 181M/181M [00:12<00:00, 14.6MB/s] 

Extracting files...


Path to dataset files: C:\Users\oevlannikov\.cache\kagglehub\datasets\hardtype\rbc-economy-news\versions\5


In [20]:
import subprocess
import sys
from tqdm.auto import tqdm

# Список пакетов для проверки/установки
packages = ["openpyxl"]

# Список встроенных модулей, которые pip не должен устанавливать
BUILTIN_MODULES = {"os", "sys", "warnings", "math", "time", "json", "re", "collections", "itertools"}

installed = []
failed = {}
skipped = []

with tqdm(total=len(packages), desc="Установка пакетов") as pbar:
    for package in packages:
        # 1. Проверяем, не встроенный ли это модуль
        if package in BUILTIN_MODULES:
            skipped.append(package)
            pbar.set_postfix_str(f"Пропущен (встроенный): {package}")
            pbar.update(1)
            continue
            
        pbar.set_postfix_str(f"Установка: {package}...")
        
        # 2. Запуск установки во внешнем процессе
        process = subprocess.Popen(
            [sys.executable, "-m", "pip", "install", package, "--progress-bar", "off"],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )
        stdout, stderr = process.communicate()
        
        # 3. Сбор результатов
        if process.returncode == 0:
            installed.append(package)
        else:
            # Отрезаем лишний системный текст pip, оставляя суть ошибки
            clean_error = "\n".join([line for line in stderr.split('\n') if line.strip()][:6])
            failed[package] = clean_error or "Неизвестная ошибка сборки"
            
        pbar.update(1)

# Итоговый честный отчет
print("\n" + "="*40 + "\n📊 ИТОГИ УСТАНОВКИ:\n" + "="*40)
if installed:
    print(f"✅ Успешно проверены/установлены ({len(installed)}): {', '.join(installed)}")
if skipped:
    print(f"ℹ️ Пропущены встроенные модули ({len(skipped)}): {', '.join(skipped)}")
if failed:
    print(f"❌ Ошибки при установке ({len(failed)}):")
    for pkg, err in failed.items():
        print(f"\n--- [{pkg}] ---")
        print(err)


Установка пакетов:   0%|          | 0/1 [00:00<?, ?it/s]


📊 ИТОГИ УСТАНОВКИ:
✅ Успешно проверены/установлены (1): openpyxl


In [5]:
import numpy as np
import pandas as pd

# 1. Читаем ваш текущий Excel файл
file_name = 'ren_tv_dataset.xlsx'
df = pd.read_excel(file_name)

print(f"Файл прочитан. Строк для обработки: {len(df)}")

# Фиксируем генератор случайных чисел
np.random.seed(42)

# 2. Генерируем логнормальное распределение
# Параметры подобраны так, чтобы основная масса была от десятков до ~2000-2500 просмотров
mu, sigma = 6.0, 1.2
raw_values = np.random.lognormal(mean=mu, sigma=sigma, size=len(df))

# Масштабируем и сдвигаем, чтобы минимальное число просмотров начиналось от ~20
# Округляем до целых чисел
simulated_views = np.clip(raw_values + 20, 20, 15000).astype(int)

# Записываем новые просмотры в датафрейм
df['views'] = simulated_views

# 3. Перезаписываем Excel файл
df.to_excel(file_name, index=False)

print(f"Новые реалистичные просмотры сгенерированы и сохранены в '{file_name}'")

# Проверим распределение: выведем минимумы, максимумы и среднее
print(f"\nМинимум просмотров в таблице: {df['views'].min()}")
print(f"Медиана просмотров: {df['views'].median():.0f}")
print(f"Максимум просмотров (топ-хайп): {df['views'].max()}")

# Выведем первые 10 строк для визуальной проверки
df.head(10)


Файл прочитан. Строк для обработки: 969
Новые реалистичные просмотры сгенерированы и сохранены в 'ren_tv_dataset.xlsx'

Минимум просмотров в таблице: 28
Медиана просмотров: 438
Максимум просмотров (топ-хайп): 15000


,url,title,views
0,https://ren.tv/news/v-rossii/1462617-medvezhon...,Медвежонок удивил автомобилистов своими трюкам...,752
1,https://ren.tv/news/sport/1439483-isu-dopustil...,ISU допустил российских фигуристов до соревнов...,361
2,https://ren.tv/news/politika/1429044-putin-i-s...,Путин и Си Цзиньпин начали переговоры в Пекине,897
3,https://ren.tv/news/politika/1462347-briussel-...,Брюссель требует от европейцев опустошить свои...,2528
4,https://ren.tv/news/proisshestviia/1462369-pia...,Пять человек погибли при взрыве на заводе по п...,324
5,https://ren.tv/news/proisshestviia/1462369-pia...,Пять человек погибли при взрыве на заводе по п...,324
6,https://ren.tv/news/sport/1447447-nedopusk-sbo...,Недопуск сборной России по хоккею до ЧМ-2027 о...,2703
7,https://ren.tv/news/v-rossii/1462335-usilenie-...,Запад забил тревогу из-за ударов ВС РФ по дата...,1033
8,https://ren.tv/news/politika/1462347-briussel-...,Брюссель требует от европейцев опустошить свои...,249
9,https://ren.tv/news/v-rossii/1462319-proshchan...,Прощание с критиком Соседовым пройдет 30 сентября,793
